[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/07_phase4_stage6_real_data.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/aditya-siraskar/PINN_FTRE/blob/main/notebooks/07_phase4_stage6_real_data.ipynb)

# 07 · Stage 6: real data, non-linear portfolio and backtests

**Phase:** Phase 4: empirical

**Goal:** Rolling one-day-ahead VaR/ES on a 5-stock NIFTY basket and a 5-ETF S&P sector basket (2006–2026; test from 2018), for a linear and a non-linear (basket + short put) book; the Stage 5 amortised PINN against Gaussian, HS, filtered HS, GARCH-t, GARCH-EVT, DCC-GARCH-t and the unfiltered Merton model; Kupiec, Christoffersen, FZ0, Diebold–Mariano (RQ2).

**Outputs:** `figures/stage6_realdata_<market>/fig01–fig05`, `results/stage6_realdata_<market>/`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive (Colab) or /kaggle/working (Kaggle), install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # Colab: keep figures/results across sessions (recommended). Ignored on Kaggle.
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
ON_KAGGLE = os.path.exists("/kaggle/working")   # needs Settings > Internet on, Accelerator > GPU T4 x2
if ON_KAGGLE:
    BASE = "/kaggle/working"   # kept with the notebook's output when run via Save Version
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
    if not os.path.exists(ROOT):
        raise RuntimeError("git clone failed: no internet. Kaggle: Settings > Internet on (phone-verified account), "
                           "then Run > Restart & clear outputs and run again.")
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files), run:")
        print(f'!git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "none (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Settings > Accelerator > GPU T4 x2)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Method in one paragraph
**Model.** The 5-asset common-jump Merton model with a daily vol scale s_t (EWMA of whitened returns, data up to
t−1): covariance s_t²Σ, jumps λ, μ_J, σ_J with equal loadings (a market-wide jump). This is exactly the density the
Stage 5 PINN learns, with θ_t = (s_t, λ_t, μ_J,t).
**Calibration.** Maximum likelihood on 2006–2017 (standard errors from the Hessian); over the test period (λ, μ_J)
are refitted monthly on a 1000-day window with shrinkage to the development fit.
**PINN.** Trained once per market on the development fit, over a θ box set from development data only. Every
test day is one query: proposal draws at θ_t, PINN weights, VaR/ES of both books. The same draws weighted by the
exact density give "Merton, filtered (exact)", so the gap between the two is the PINN error alone.
**Books.** Linear: equal-weight basket. Non-linear: basket + short one 3%-OTM one-month put on the basket
(Black–Scholes, implied vol = EWMA basket vol, rolled monthly), revalued in full on every scenario.

Data: `data/processed/<market>_returns.csv` (committed snapshot of Yahoo prices), so reruns give the same numbers.

In [ ]:
print(open('configs/stage6_realdata.yaml').read())

## 2. Calibrate, train and backtest (NIFTY, S&P)
Resumable: finished markets are skipped. Estimated (not yet measured) **about 30–40 min per market on a T4**
(PINN training ≈ 25 min; calibration, GARCH/DCC filters and ~2200 daily forecasts the rest).
A crash after training can be re-evaluated without retraining: add `--eval-only`.

In [ ]:
import pathlib
STEPS = 30000
for market in ("nifty", "sp"):
    run = f"stage6_realdata_{market}"
    if pathlib.Path(f"results/{run}/metrics.json").exists():
        print("skip", run); continue
    !python scripts/stage6_realdata.py --market {market} --steps {STEPS}

# d = 10 basket: not pursued (Stage 4 breakdown point). Skeleton for future work:
# add a 10-ticker market to configs/stage6_realdata.yaml, then
# !python scripts/stage6_realdata.py --market nifty10 --steps 60000

## 3. Figures and what to check

| Figure | Pass criterion |
|---|---|
| fig01_training | loss terms fall; tracked 1-day ES₉₉ error at the development market ≲ 1% |
| fig02_data_calibration | s_t stays inside the PINN box (clipped days are counted in metrics); rolling λ, μ_J stable, near the development fit |
| fig03_var_paths | VaR₉₉ follows volatility (filtered models) through COVID; unfiltered models react slowly and breach in clusters |
| fig04_backtests | breach rates inside the Kupiec region; no ×; FZ0 bars show which models beat the PINN (negative) and whether significantly |
| fig05_pinn_vs_exact | daily PINN/exact ES₉₉ gap ≲ 1–2% on both books (the PINN reproduces its own model on real states) |

In [ ]:
for market in ("nifty", "sp"):
    print(f"======== {market}")
    show_figures(f"stage6_realdata_{market}")

## 4. Numbers

In [ ]:
for market in ("nifty", "sp"):
    m = load_metrics(f"stage6_realdata_{market}")
    print(f"== {market}: test {m['test']} ({m['n_test']} days); dev fit lam={m['dev_fit']['lam']:.1f} "
          f"mu_J={m['dev_fit']['mu_j']:.4f} sigma_J={m['dev_fit']['sigma_j']:.4f}; clipped days {m['n_clipped']}; "
          f"train {m['train_time_s'] / 60:.0f} min")
    print("   PINN vs exact ES:", {k: round(100 * v["median_abs"], 2) for k, v in m["pinn_vs_exact"].items()}, "% (median |gap|)")
    bt = pd.read_csv(f"results/stage6_realdata_{market}/backtest.csv")
    cols = ["model", "breaches", "rate", "p_uc", "p_cc", "fz0", "dm_p_vs_pinn", "es_ratio"]
    for book in ("linear", "nonlinear"):
        for a in (0.99, 0.975):
            print(f"   {book}, alpha = {a}")
            display(bt[(bt.book == book) & (bt.alpha == a)][cols].sort_values("fz0").style.format(precision=4).hide(axis="index"))